# Huấn luyện detector trên Kaggle

Chỉnh cấu hình ở mục 2-3 ngay trong notebook (không cần sửa `settings.py`), rồi chạy lần lượt các cell.

| | |
|---|---|
| **Đầu vào** | Dataset `vn-av-df-data-partN` + cache `vn-av-df-features-partN` từ `prepare.ipynb` (không tải encoder, không cài worker) |
| **Model** | `"all"` (8 kiến trúc) hoặc danh sách kiến trúc tuỳ chọn; mỗi kiến trúc chạy các seed đã chọn |
| **Báo cáo** | Riêng từng detector/seed trên validation, rồi **test chạy ngay sau train** (`RUN_TEST = True`). So sánh nhiều kiến trúc ở `compare.ipynb` |
| **Kaggle** | GPU (khuyên dùng) hoặc CPU; không cần Internet ngoài `git clone`/`pip` |

## 1. Cài đặt
Clone repository, cài `data_pipeline` và gói `[training]`.

In [ ]:
from pathlib import Path
import subprocess, sys, os
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-df-forensics.git"  # Sửa nếu đổi repository.
ROOT = Path("/kaggle/working/vn-av-df-forensics")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
assert (ROOT / "settings.py").is_file(), "Cần upload/push code mới trước khi dùng notebook"
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT / "data_pipeline"), "-e", str(ROOT) + "[training]"], check=True)
import settings
settings.bootstrap()
from vn_av_df.actions import execute

## 2. Chọn part, kiến trúc, cache và tên lần chạy
- `DATA_PARTS`: đường dẫn đầy đủ từng part (phải trùng lúc prepare), `ARCHITECTURES`: `"all"` hoặc danh sách kiến trúc.
- `CACHE_INPUTS`: thư mục cache (dataset `vn-av-df-features-partN`); nhiều thư mục được gộp (vd. FATE và AV-HuBERT+DINOv2 trích ở 2 phiên).
- `RUN_NAME` là folder output, ví dụ `train_part1_run01`, `train_parts1_2_run01`.
- Đổi dữ liệu/model/hyperparameter thì dùng tên run mới. `RESUME` chỉ dùng khi tiếp tục đúng lần chạy bị ngắt.
- `RUN_TEST = True`: test một lần ngay sau train với checkpoint/ngưỡng chọn trên validation (khoá bằng `test-lock.json`).

In [ ]:
# Mỗi part một đường dẫn đầy đủ (copy từ panel Input): thư mục có dataset_info.json,
# hoặc gốc dataset chứa đúng một thư mục như vậy. Nhiều part: liệt kê hết.
DATA_PARTS = [Path("/kaggle/input/datasets/YOUR_USER/vn-av-df-data-part1")]

ARCHITECTURES = "all"  # Cả 8 kiến trúc trong ALL_METHODS (mục 4).
# ARCHITECTURES = ["fate_gru", "avh_tcn", "p2_syncartifact"]  # B-FATE, B-AVH, P2.
# ARCHITECTURES = ["p2_sync_only", "p2_artifact_only", "p2_concat", "p2_sync_seen_fake", "avh_realrecon"]
# ARCHITECTURES = ["fate_gru"]  # Chỉ B-FATE: chỉ cần cache fate/.
# Kiến trúc AV-HuBERT/P2 cần cache avhubert/ (+ dinov2/ với p2_*).
RUN_NAME = "train_part1_run01"
DEVICE = "cuda"
SEEDS = [42, 43, 44]  # Pilot: [42]. 3 kiến trúc x 3 seed = 9 lượt train detector.
RESUME = False
RUN_TEST = True  # Test ngay sau train; không dùng test để chọn model.
# Cache từ prepare.ipynb: thư mục chứa fate/, avhubert/, dinov2/ (có thể ở nhiều dataset).
CACHE_INPUTS = [Path("/kaggle/input/datasets/YOUR_USER/vn-av-df-features-part1")]
CACHE_DIR = Path("/kaggle/temp/vn-av-df-cache")  # Chỉ chứa symlink tới CACHE_INPUTS, không vào Output.
HELD_OUT_GENERATORS = ["musetalk_1_5"]  # Generator chỉ dùng test trong protocol chính.
# Pilot chỉ có Wav2Lip: đặt [] (đây là protocol khác; dùng RUN_NAME riêng).

## 3. Thông số train
- `TRAINING` áp dụng cho detector được chọn; `RECONSTRUCTION` (stage A) và `SYNC` (stage S) chỉ dùng cho nhánh tương ứng.
- Linear không dùng hidden/dropout; training xử lý một clip mỗi bước tối ưu, không có batch size detector.
- Thông số trích feature nằm ở `prepare.ipynb`; train dùng đúng cache đã trích.

In [ ]:
TRAINING = {
    "epochs": 20,             # Pilot: 2.
    "lr": 3e-4,              # Learning rate AdamW.
    "weight_decay": 1e-4,    # Regularization.
    "hidden": 128,           # Kích thước hidden/projection của temporal head.
    "dropout": 0.1,
    "patience": 5,           # Dừng sau 5 epoch validation loss không cải thiện; 0 = tắt.
    "top_fraction": 0.1,     # Score video = mean top 10% ô hợp lệ.
    "torch_threads": 2,      # Luồng CPU, không phải batch size.
    "balance_parents": True, # Mỗi epoch lấy một variant/parent để cân bằng nguồn.
    "artifact_aux_weight": 0.5,  # P2: trọng số head phụ artifact (nhãn AI theo ô).
}
RECONSTRUCTION = {
    "epochs": 20,            # Pilot: 2. Stage A của P1/P2: chỉ real train/validation.
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "patience": 5,
}
SYNC = {
    "epochs": 20,            # Pilot: 2. Stage S của P2: real, sham, real dịch lệch; không thấy fake.
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "patience": 5,
    "shift_frames": [3, 15],  # Dịch audio ±3-15 frame (0,12-0,6 s) để tạo mẫu lệch.
    "shift_probability": 0.5, # Tỷ lệ real được dịch lệch mỗi epoch.
}
BOOTSTRAP_REPLICATES = 1000  # CI khi chủ động chạy test; không ảnh hưởng train.

## 4. Áp cấu hình và gắn cache
Kiểm tra dữ liệu (hash từng part), rồi gắn cache (symlink, không chép) và kiểm cache đủ encoder các kiến trúc cần.

In [ ]:
ALL_METHODS = [
    "fate_gru", "avh_tcn", "p2_syncartifact",
    "p2_sync_only", "p2_artifact_only", "p2_concat", "p2_sync_seen_fake", "avh_realrecon",
]
chosen = ALL_METHODS.copy() if ARCHITECTURES == "all" else ARCHITECTURES
if not isinstance(chosen, list) or not chosen or len(set(chosen)) != len(chosen):
    raise ValueError("ARCHITECTURES phải là 'all' hoặc danh sách model không trùng")
from vn_av_df.dataset import part_selection
settings.DATASET, settings.DATASET_PARTS = part_selection(DATA_PARTS)
settings.ARCHITECTURES, settings.SEEDS = chosen, SEEDS
settings.RUN_NAME, settings.DEVICE = RUN_NAME, DEVICE
settings.DEMO_METHOD = chosen[0]  # Chỉ làm mặc định demo, không phải model tốt nhất.
cfg = settings.config()
if set(chosen) - set(cfg["methods"]):
    raise ValueError("Tên model không có trong settings.config()['methods']")
cfg["cache"] = str(CACHE_DIR)
cfg["held_out_generators"] = HELD_OUT_GENERATORS
cfg["training"].update(TRAINING)
cfg["reconstruction"].update(RECONSTRUCTION)
cfg["sync"].update(SYNC)
cfg["bootstrap_replicates"] = BOOTSTRAP_REPLICATES
from vn_av_df.dataset import training_dataset
rows, selection = training_dataset(cfg, verify_media=True)
print("Part và hash:", selection)
print("Model:", chosen, "| Seeds:", SEEDS, "| Output:", cfg["runs"])

In [ ]:
import json
from vn_av_df.experiment import attach_caches
found = attach_caches(CACHE_INPUTS, CACHE_DIR)
# Encoder các kiến trúc đã chọn cần; cache phải trích xong trên đúng dữ liệu (hash manifest).
needed = sorted({key for name in chosen for key in (cfg["methods"][name]["encoder"], cfg["methods"][name].get("artifact_encoder")) if key})
for key in needed:
    info = CACHE_DIR / key / "extraction.json"
    if not info.exists():
        raise FileNotFoundError(f"Thiếu cache {key} (đã thấy: {sorted(found)}): chạy prepare.ipynb với ENCODERS chứa '{key}' "
                                "hoặc thêm thư mục cache đó vào CACHE_INPUTS")
    done = json.loads(info.read_text(encoding="utf-8"))
    if done["dataset_selection"]["manifest_sha256"] != selection["manifest_sha256"]:
        raise ValueError(f"Cache {key} trích trên dữ liệu khác DATASET_PARTS; trích lại bằng prepare.ipynb")
    print(f"Cache {key}: {done['samples']} mẫu từ {found[key]}")

## 5. W&B (tuỳ chọn)
Mỗi bước một run, cùng group = `RUN_NAME`:
- `<RUN_NAME>-<detector>_seed<N>`: ghi **theo từng epoch** lúc train (stage A, stage S, detector: loss, AUC, F1, AUC theo ô 2×2, AUC từng nhánh P2, thời gian epoch) cùng log console và GPU/RAM.
- Mục 9 bổ sung metric validation/test, bảng theo nhóm, dự đoán từng video, ROC/PR và ảnh vào đúng run detector.
- Lần đầu: thêm Kaggle secret `WANDB_API_KEY` (*Add-ons → Secrets*, bật cho notebook).
- Tắt: đặt `USE_WANDB = False`. Không có secret thì tự tắt, pipeline vẫn chạy.

In [ ]:
import json
USE_WANDB = True
WANDB_PROJECT = "vn-av-df-forensics"
WANDB_ENTITY = None  # Team W&B; None = tài khoản của API key.
if USE_WANDB:
    # Log W&B ở /kaggle/working/wandb (ngoài repo): còn trong Output, `wandb sync` lại được.
    os.environ.setdefault("WANDB_DIR", "/kaggle/working")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb
    try:
        from kaggle_secrets import UserSecretsClient
        # Key qua biến môi trường (không truyền vào code); W&B tự đọc.
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
        wandb.login()
    except Exception as exc:  # Không có secret: tắt W&B, pipeline vẫn chạy.
        USE_WANDB = False
        print("Tắt W&B, không đọc được secret WANDB_API_KEY:", exc)

def plain(value):
    """Cấu hình/metric về kiểu JSON đơn giản để W&B nhận."""
    return json.loads(json.dumps(value, default=str))

def stamp():
    """Ngày giờ Việt Nam (máy Kaggle chạy UTC) gắn vào tên run để phân biệt các lần chạy."""
    from datetime import datetime, timedelta, timezone
    return datetime.now(timezone(timedelta(hours=7))).strftime("%m%d-%H%M")

def table(rows):
    """Bảng W&B từ list dict; ô lồng (dict/list) đổi thành chuỗi JSON."""
    import pandas as pd
    frame = pd.DataFrame(rows)
    for column in frame.columns:
        if frame[column].map(lambda v: isinstance(v, (dict, list))).any():
            frame[column] = frame[column].map(lambda v: json.dumps(v, ensure_ascii=False))
    return wandb.Table(dataframe=frame)

cfg.pop("wandb", None)
if USE_WANDB:  # Code train đọc cfg["wandb"] để ghi theo từng epoch (mỗi detector/seed một run).
    cfg["wandb"] = {"project": WANDB_PROJECT, "entity": WANDB_ENTITY, "group": RUN_NAME}

## 6. Train
Phiên bị ngắt: giữ run, chạy lại với `RESUME = True`, cùng dữ liệu/cấu hình/code/cache.

In [ ]:
execute("train", cfg, resume=RESUME)  # Chỉ train, không compare và không tự evaluate.

## 7. Báo cáo từng detector (validation)
- Loss/validation theo epoch; ROC, PR, confusion matrix và phân bố score của best checkpoint.
- Timeline dùng một real và một fake đầu tiên theo sample ID, không chọn theo độ đẹp của dự đoán.
- Validation đã dùng chọn checkpoint/ngưỡng, nên không coi là kết quả test độc lập.
- Chạy lại được riêng cell này từ thư mục artifact đã tải: chỉ cần JSON và thư viện vẽ, không cần GPU/video.

In [ ]:
from IPython.display import display, Markdown, Image
from vn_av_df.reporting import training_reports

def fmt(value):
    return f"{value:.4f}" if isinstance(value, float) else ("-" if value is None else str(value))

def show_detector_reports(split="validation"):
    # Mỗi vòng là một detector/seed, không tạo bảng so sánh nhiều detector.
    for item in training_reports(cfg["runs"], split=split):
        info = item["summary"]
        display(Markdown(f"### {info['architecture']} - seed {info['seed']} - {split}"))
        display(Markdown(f"Best epoch: **{info['best_epoch']}** / {info['epochs_completed']} epoch; "
                         f"video coverage: **{info['video_coverage']:.1%}**."))
        lines = ["| Chỉ số video | Giá trị |", "|---|---|"]
        for name, value in info["video_metrics"].items():
            if name != "confusion_matrix":
                lines.append(f"| {name} | {fmt(value)} |")
        display(Markdown("\n".join(lines)))
        # Ô 2×2: fake tính cùng real gốc; real/sham riêng chỉ có ý nghĩa ở false_alarm_rate.
        lines = ["| Ô | N | ROC-AUC | F1 | False alarm |", "|---|---|---|---|---|"]
        for cell, m in info["by_condition"].items():
            lines.append(f"| {cell} | {m['samples']} | {fmt(m['roc_auc'])} | {fmt(m['f1'])} | {fmt(m['false_alarm_rate'])} |")
        display(Markdown("\n".join(lines)))
        if info.get("branches"):
            # P2: AUC từng nhánh, real gốc (0) so với từng ô (1); dòng sham = độ nhạy lệch A/V.
            scores = info["branches"]["scores"]
            cells = sorted({c for v in scores.values() for c in v})
            lines = ["| Nhánh | " + " | ".join(cells) + " |", "|---" * (len(cells) + 1) + "|"]
            for name, values in scores.items():
                lines.append(f"| {name} | " + " | ".join(fmt(values.get(c, {}).get("roc_auc")) for c in cells) + " |")
            display(Markdown("\n".join(lines)))
        display(Markdown(f"Trainable parameters: **{info['resources']['trainable_parameters']:,}**; "
                         f"thời gian train trong phiên: **{info['resources']['elapsed_this_session_s']:.1f}s**."))
        if info.get("reconstruction"):
            display(Markdown(f"Stage A→V real-only - best validation loss: "
                             f"**{info['reconstruction']['best_validation_loss']:.4f}**; "
                             f"constant predictor: **{info['reconstruction']['constant_validation_loss']:.4f}**."))
        if info.get("sync"):
            display(Markdown(f"Stage S sync (không thấy fake) - best validation loss: "
                             f"**{info['sync']['best_validation_loss']:.4f}**; "
                             f"AUC real vs sham/dịch lệch: **{fmt(info['sync']['best_validation_auc'])}**."))
        for path in item["images"]:
            display(Image(filename=path))

show_detector_reports("validation")

## 8. Test
Khi `RUN_TEST = True` (mặc định), đánh giá riêng từng detector một lần với checkpoint/ngưỡng đã khóa trên validation.
Không xếp hạng hay so sánh cặp; lưu `test.json`, `predictions-test.json`, ảnh từng detector, `evaluation.json` và `test-lock.json`.

In [ ]:
if RUN_TEST:
    execute("evaluate", cfg)
    show_detector_reports("test")
else:
    print("Chưa chạy test. Báo cáo bên trên là validation.")

## 9. Bổ sung kết quả lên W&B
Mở lại run của từng detector/seed, thêm: kết quả stage A/S, metric best checkpoint (validation) và test (nếu có), temporal AP, ma trận nhầm lẫn,
bảng theo ô 2×2 / nhánh P2 / generator / variant / part, bảng dự đoán từng video, ROC/PR tương tác và ảnh báo cáo. Bỏ qua nếu `USE_WANDB = False`.

In [ ]:
# Bổ sung vào run của từng detector/seed (đã ghi theo epoch lúc train): kết quả stage A/S, metric
# best checkpoint và test (nếu RUN_TEST), bảng theo ô 2×2 / nhánh P2 / generator / variant / part,
# bảng dự đoán từng video, ROC/PR tương tác, ma trận nhầm lẫn và ảnh báo cáo.
if globals().get("USE_WANDB"):  # Bỏ qua cell mục 5 cũng coi như tắt W&B.
    runs = Path(cfg["runs"])

    def no_matrix(metrics):
        return {k: v for k, v in metrics.items() if k != "confusion_matrix"}

    for entry in json.loads((runs / "training.json").read_text(encoding="utf-8"))["runs"]:
        folder = runs / f"{entry['architecture']}_seed{entry['seed']}"

        def load(name):
            path = folder / name
            return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

        live = load("wandb.json")  # Run đã ghi theo epoch lúc train.
        if live:
            run = wandb.init(project=live["project"], entity=live["entity"] or None, id=live["id"], resume="must")
        else:  # Train khi tắt W&B: tạo run mới, ghi lịch sử từ file.
            run = wandb.init(
                project=WANDB_PROJECT, entity=WANDB_ENTITY, job_type="train", group=RUN_NAME,
                name=f"{RUN_NAME}-{folder.name}-{stamp()}",
                config=plain({"architecture": entry["architecture"], "seed": entry["seed"],
                              "dataset": selection, "training": TRAINING}),
            )
            # Mỗi stage có trục epoch riêng: stageA (tái dựng A→V), stageS (sync), detector.
            for stage, name in (("stageA", "reconstruction.json"), ("stageS", "sync.json"),
                                ("detector", "history.json")):
                data = load(name)
                history = data if isinstance(data, list) else (data or {}).get("history", [])
                run.define_metric(f"{stage}/epoch")
                run.define_metric(f"{stage}/*", step_metric=f"{stage}/epoch")
                for row in history:
                    run.log({f"{stage}/{k}": v for k, v in row.items() if v is not None})
        best = load("best.json") or {}
        run.summary.update(plain({"best_epoch": best.get("epoch"), "thresholds": best.get("thresholds")}))
        # Stage A: R tốt hơn dự đoán hằng số bao nhiêu; stage S: AUC real vs sham/dịch lệch.
        stage_a, stage_s = load("reconstruction.json"), load("sync.json")
        if stage_a:
            run.summary.update({"stageA/best_validation_loss": stage_a["best_validation_loss"],
                                "stageA/constant_validation_loss": stage_a["constant_validation_loss"],
                                "stageA/real_train": len(stage_a["train_ids"]),
                                "stageA/real_validation": len(stage_a["validation_ids"])})
        if stage_s:
            run.summary.update(plain({"stageS/best_validation_loss": stage_s["best_validation_loss"],
                                      "stageS/best_validation_auc": stage_s["best_validation_auc"],
                                      "stageS/sham_train": stage_s["sham_train"],
                                      "stageS/sham_validation": stage_s["sham_validation"]}))
        thresholds = best.get("thresholds") or {}
        for split in ("validation", "test"):  # test chỉ có khi RUN_TEST=True.
            metrics = load(f"{split}.json")
            if not metrics:
                continue
            video = metrics["video"]
            run.summary.update(plain({f"{split}/{k}": v for k, v in no_matrix(video).items()}))
            run.summary.update(plain({f"{split}/{k}": metrics.get(k) for k in (
                "video_coverage", "window_coverage", "temporal_labeled_videos",
                "real_video_interval_false_alarm_rate", "false_intervals_per_real_hour",
                "head_inference_seconds")}))
            for iou_threshold, values in (metrics.get("temporal") or {}).items():
                run.summary.update(plain({f"{split}/temporal_iou{iou_threshold}/{k}": v for k, v in values.items()}))
            (tn, fp), (fn, tp) = video["confusion_matrix"]
            run.summary.update({f"{split}/tn": tn, f"{split}/fp": fp, f"{split}/fn": fn, f"{split}/tp": tp})
            if metrics.get("uncertainty"):
                run.summary.update(plain({f"{split}/uncertainty": metrics["uncertainty"]}))
            # Bảng theo nhóm: ô 2×2 (mọi split), generator/variant/part (chỉ test).
            for key in ("by_condition", "by_generator", "by_variant", "by_part"):
                groups = metrics.get(key) or {}
                if groups:
                    run.log({f"{split}/{key}": table([{"group": g, **no_matrix(m["video"])} for g, m in groups.items()])})
            scores = (metrics.get("branches") or {}).get("scores") or {}
            branch_rows = [{"branch": b, "cell": c, **v} for b, values in scores.items() for c, v in values.items()]
            if branch_rows:
                run.log({f"{split}/branches": table(branch_rows)})
            # Dự đoán từng video (không gồm timeline) + ROC/PR tương tác để xem lỗi theo mẫu/ô.
            predictions = load(f"predictions-{split}.json") or []
            measured = [p for p in predictions if p["video_score"] is not None]
            if measured:
                run.log({f"{split}/predictions": table([
                    {k: p.get(k) for k in ("sample_id", "label", "condition", "dataset_part", "source_id",
                                           "duration_s", "video_score")}
                    | {"predicted": int(p["video_score"] >= thresholds.get("video", 0.5))}
                    for p in measured])})
                labels = [p["label"] for p in measured]
                if len(set(labels)) == 2:
                    probas = [[1 - p["video_score"], p["video_score"]] for p in measured]
                    run.log({f"{split}/roc": wandb.plot.roc_curve(labels, probas, labels=["real", "fake"], classes_to_plot=[1]),
                             f"{split}/pr": wandb.plot.pr_curve(labels, probas, labels=["real", "fake"], classes_to_plot=[1])})
        resources = load("resources.json") or {}
        run.summary.update(plain({f"resources/{k}": v for k, v in resources.items()
                                  if isinstance(v, (int, float))}))
        images = sorted((folder / "report").glob("*.png"))  # Có sau khi chạy cell báo cáo.
        if images:
            run.log({"report": [wandb.Image(str(p), caption=p.stem) for p in images]})
        run.finish()

## 10. Dọn Output
Output chỉ giữ `vn-av-df-forensics/runs/<RUN_NAME>/` (checkpoint, cấu hình, part/hash, dự đoán, hình báo cáo, test) và log W&B; code repo bị xoá, cache nằm ở dataset input (symlink trong /kaggle/temp).
*Save Version*, tải `runs/<RUN_NAME>` (`tools/kaggle_pull.py ... "/runs/"`) rồi so sánh bằng `compare.ipynb`.

In [ ]:
from vn_av_data.common.runtime import prune_output
runs = Path(cfg["runs"])

def size_gb(path):
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 1e9 if path.exists() else 0.0

prune_output(ROOT, [runs.relative_to(ROOT)])
print(f"Run: {runs} ({size_gb(runs):.2f} GB)")
print(f"Tổng /kaggle/working: {size_gb(Path('/kaggle/working')):.2f} GB (giới hạn Output 20 GB)")